In [4]:
import pandas as pd
import requests
import time
import json
import logging
import os
from pathlib import Path

# 设置日志记录
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DeepSeekSentimentAnalyzer:
    def __init__(self, api_key, base_url="https://api.deepseek.com/v1"):
        """
        初始化DeepSeek API客户端
        :param api_key: DeepSeek API密钥
        :param base_url: API基础URL
        """
        self.api_key = api_key
        self.base_url = base_url
        self.headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }
    
    def analyze_sentiment(self, comment, max_retries=3):
        """
        使用DeepSeek模型分析评论情感
        :param comment: 评论内容
        :param max_retries: 最大重试次数
        :return: 情感分析结果
        """
        # 构建系统提示词
        system_prompt = """你是一个专业的产品评论情感分析专家。请仔细分析用户评论的情感倾向。
        
        分析标准：
        1. 正面/种草：用户表达积极态度、推荐、满意、称赞、购买意向、使用体验好
        2. 中立/观望：用户表达中立观点、询问信息、客观描述、无明确倾向、利弊都有提及
        3. 负面/拔草：用户表达消极态度、批评、不满意、不推荐、问题反馈、质量抱怨
        
        重要提示：
        - 请全面理解上下文语义，不要仅根据关键词判断
        - 考虑评论的整体语气和意图
        - 对于复杂评论，综合考虑所有方面
        - 确保判断准确合理，多思考检查一下
        
        输出格式要求：
        请只输出以下三种结果之一（不要添加任何其他文字、标点或解释）：
        正面/种草
        中立/观望
        负面/拔草"""
        
        # 构建用户消息
        user_message = f"请分析以下用户评论的情感倾向：\n\n评论内容：{comment}"
        
        # 构建请求数据
        data = {
            "model": "deepseek-chat",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            "max_tokens": 50,
            "temperature": 0.1,  # 低温确保结果一致
            "stream": False
        }
        
        for attempt in range(max_retries):
            try:
                # 发送API请求
                response = requests.post(
                    f"{self.base_url}/chat/completions",
                    headers=self.headers,
                    json=data,
                    timeout=30
                )
                
                # 检查响应状态
                if response.status_code == 200:
                    result = response.json()
                    sentiment = result["choices"][0]["message"]["content"].strip()
                    
                    # 验证和清理输出
                    valid_sentiments = ["正面/种草", "中立/观望", "负面/拔草"]
                    
                    # 精确匹配
                    if sentiment in valid_sentiments:
                        return sentiment
                    
                    # 模糊匹配
                    if "正面" in sentiment or "种草" in sentiment:
                        return "正面/种草"
                    elif "负面" in sentiment or "拔草" in sentiment:
                        return "负面/拔草"
                    else:
                        # 默认返回中立
                        return "中立/观望"
                
                elif response.status_code == 429:
                    # 速率限制，等待后重试
                    wait_time = 2 ** attempt  # 指数退避
                    logger.warning(f"速率限制，等待 {wait_time} 秒后重试 (尝试 {attempt + 1}/{max_retries})")
                    time.sleep(wait_time)
                    continue
                    
                else:
                    logger.error(f"API请求失败，状态码: {response.status_code}, 响应: {response.text}")
                    if attempt < max_retries - 1:
                        time.sleep(1)
                        continue
            
            except requests.exceptions.Timeout:
                logger.warning(f"请求超时，重试中 (尝试 {attempt + 1}/{max_retries})")
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
            
            except Exception as e:
                logger.error(f"API调用异常: {str(e)}")
                if attempt < max_retries - 1:
                    time.sleep(1)
                    continue
        
        # 所有重试都失败，返回默认值
        logger.error(f"所有重试失败，评论: {comment[:50]}...")
        return "中立/观望"

def find_excel_file(directory, filename="有效数据.xlsx"):
    """
    在目录中查找Excel文件
    """
    dir_path = Path(directory)
    if not dir_path.exists():
        logger.error(f"目录不存在: {directory}")
        return None
    
    # 尝试精确匹配
    file_path = dir_path / filename
    if file_path.exists():
        return str(file_path)
    
    # 搜索所有Excel文件
    excel_files = list(dir_path.glob("*.xlsx")) + list(dir_path.glob("*.xls"))
    
    if not excel_files:
        logger.error(f"在目录 {directory} 中找不到Excel文件")
        return None
    
    # 显示找到的文件
    logger.info(f"在目录中找到以下Excel文件:")
    for i, file in enumerate(excel_files, 1):
        logger.info(f"  {i}. {file.name}")
    
    # 如果有多个文件，让用户选择
    if len(excel_files) == 1:
        return str(excel_files[0])
    else:
        # 这里可以选择第一个文件，或者添加交互选择逻辑
        logger.info(f"选择第一个文件: {excel_files[0].name}")
        return str(excel_files[0])

def process_comments_batch(analyzer, comments, batch_size=10, delay=1):
    """
    批量处理评论
    :param analyzer: DeepSeekSentimentAnalyzer实例
    :param comments: 评论列表
    :param batch_size: 每批处理数量
    :param delay: 批次间延迟(秒)
    :return: 情感分析结果列表
    """
    sentiments = []
    total_comments = len(comments)
    
    for i in range(0, total_comments, batch_size):
        batch_end = min(i + batch_size, total_comments)
        batch_comments = comments[i:batch_end]
        
        logger.info(f"处理批次 {i//batch_size + 1}/{(total_comments + batch_size - 1)//batch_size}")
        
        for j, comment in enumerate(batch_comments):
            index = i + j
            # 处理空评论
            if pd.isna(comment) or str(comment).strip() == "":
                sentiments.append("中立/观望")
                logger.info(f"  评论 {index + 1}/{total_comments}: 空评论 -> 中立/观望")
                continue
            
            comment_str = str(comment).strip()
            
            # 显示前50个字符以便跟踪
            display_comment = comment_str[:50] + "..." if len(comment_str) > 50 else comment_str
            logger.info(f"  评论 {index + 1}/{total_comments}: {display_comment}")
            
            # 分析情感
            sentiment = analyzer.analyze_sentiment(comment_str)
            sentiments.append(sentiment)
            
            logger.info(f"      结果: {sentiment}")
            
            # 单条评论间短延迟
            time.sleep(0.5)
        
        # 批次间延迟
        if batch_end < total_comments:
            logger.info(f"批次完成，等待 {delay} 秒...")
            time.sleep(delay)
    
    return sentiments

def main():
    """
    主处理函数
    """
    # 配置API密钥 - 请替换为你的DeepSeek API密钥
    API_KEY = "sk-f76855f9633d451c986d94be1b8d1a79"
    
    # 初始化分析器
    analyzer = DeepSeekSentimentAnalyzer(API_KEY)
    
    # Excel文件路径 - 根据您提供的实际位置
    base_directory = "/Users/griffinx/Desktop/Maison/2. Antigravity/上市后监测"
    
    # 尝试多个可能的子目录
    possible_subdirs = [
        "python 批量判定情感度",  # 您提供的实际位置
        "python上市后监测",  # 您之前提到的位置
        "python上市后监测/python 批量判定情感度"  # 也可能是嵌套结构
    ]
    
    file_path = None
    excel_filename = "有效数据.xlsx"
    
    # 首先检查文件是否存在
    logger.info("正在查找Excel文件...")
    
    # 尝试直接路径
    direct_path = os.path.join(base_directory, "python 批量判定情感度", excel_filename)
    if os.path.exists(direct_path):
        file_path = direct_path
        logger.info(f"找到文件: {file_path}")
    else:
        logger.info(f"直接路径不存在: {direct_path}")
        
        # 在各个子目录中查找
        for subdir in possible_subdirs:
            test_path = os.path.join(base_directory, subdir)
            logger.info(f"检查目录: {test_path}")
            
            found_file = find_excel_file(test_path, excel_filename)
            if found_file:
                file_path = found_file
                break
    
    if not file_path:
        logger.error("无法找到Excel文件，请检查路径或手动指定文件路径")
        return
    
    logger.info(f"使用文件: {file_path}")
    
    try:
        # 1. 读取Excel文件
        logger.info(f"正在读取Excel文件...")
        df = pd.read_excel(file_path)
        
        # 显示数据基本信息
        logger.info(f"数据形状: {df.shape}")
        logger.info(f"列名: {df.columns.tolist()}")
        
        # 2. 获取评论数据
        # 尝试按列名获取，如果不存在则按位置获取
        if df.shape[1] >= 2:
            # 显示前几行数据预览
            logger.info("\n前5行数据预览:")
            for i in range(min(5, len(df))):
                row = df.iloc[i]
                logger.info(f"行{i+1}: A列='{str(row.iloc[0])[:50]}...' B列='{str(row.iloc[1])[:50]}...'")
            
            # 确定使用哪一列（B列或第二列）
            if df.shape[1] > 1:
                chinese_comments = df.iloc[:, 1]  # B列（索引1）
                logger.info(f"使用第二列（索引1）作为中文评论数据")
            else:
                logger.error("Excel文件需要至少有两列数据")
                return
        else:
            logger.error("Excel文件需要至少有两列数据")
            return
        
        total_comments = len(chinese_comments)
        logger.info(f"找到 {total_comments} 条中文评论")
        
        # 3. 情感分析
        logger.info("开始情感分析...")
        
        # 确认是否继续
        response = input(f"即将分析 {total_comments} 条评论，这可能需要一些时间。继续吗？(y/n): ")
        if response.lower() != 'y':
            logger.info("用户取消操作")
            return
        
        # 批量处理
        sentiments = process_comments_batch(
            analyzer=analyzer,
            comments=chinese_comments.tolist(),
            batch_size=15,
            delay=2
        )
        
        # 4. 添加结果到DataFrame
        df['情感分析结果'] = sentiments
        
        # 5. 保存结果到新文件
        output_dir = os.path.dirname(file_path)
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        output_filename = f"{base_name}_情感分析结果.xlsx"
        output_path = os.path.join(output_dir, output_filename)
        
        df.to_excel(output_path, index=False)
        logger.info(f"结果已保存到: {output_path}")
        
        # 6. 生成统计报告
        logger.info("\n" + "="*50)
        logger.info("情感分析统计报告")
        logger.info("="*50)
        
        sentiment_counts = pd.Series(sentiments).value_counts().sort_index()
        for sentiment, count in sentiment_counts.items():
            percentage = (count / total_comments) * 100
            logger.info(f"{sentiment}: {count}条 ({percentage:.2f}%)")
        
        # 7. 验证结果完整性
        logger.info("\n验证结果:")
        logger.info(f"总评论数: {total_comments}")
        logger.info(f"分析结果数: {len(sentiments)}")
        
        if total_comments == len(sentiments):
            logger.info("✓ 所有评论都已成功分析")
        else:
            logger.warning(f"⚠ 数量不匹配！可能缺少 {total_comments - len(sentiments)} 条评论的分析结果")
        
        # 8. 显示部分结果示例
        logger.info("\n分析结果示例（前10条）:")
        for i in range(min(10, len(df))):
            comment = str(df.iloc[i, 1]) if not pd.isna(df.iloc[i, 1]) else "空评论"
            sentiment = df.iloc[i]['情感分析结果']
            comment_preview = comment[:50] + "..." if len(comment) > 50 else comment
            logger.info(f"{i+1}. 评论: {comment_preview} -> {sentiment}")
        
        logger.info(f"\n结果文件详细信息:")
        logger.info(f"  原始文件: {file_path}")
        logger.info(f"  结果文件: {output_path}")
        logger.info(f"  总行数: {len(df)}")
        logger.info(f"  新增列: '情感分析结果' (第{len(df.columns)}列)")
    
    except Exception as e:
        logger.error(f"处理过程中出错: {str(e)}")
        import traceback
        traceback.print_exc()

def test_environment():
    """
    测试环境和文件路径
    """
    logger.info("=== 环境测试 ===")
    
    # 测试文件路径
    test_paths = [
        "/Users/griffinx/Desktop/Maison/2. Antigravity/上市后监测/python 批量判定情感度",
        "/Users/griffinx/Desktop/Maison/2. Antigravity/上市后监测/python上市后监测"
    ]
    
    for path in test_paths:
        logger.info(f"\n检查路径: {path}")
        if os.path.exists(path):
            logger.info(f"✓ 路径存在")
            # 列出目录内容
            try:
                files = os.listdir(path)
                excel_files = [f for f in files if f.endswith(('.xlsx', '.xls'))]
                logger.info(f"  目录内容: {len(files)} 个文件/文件夹")
                if excel_files:
                    logger.info(f"  Excel文件: {excel_files}")
                else:
                    logger.info(f"  没有Excel文件")
            except Exception as e:
                logger.error(f"  无法读取目录: {e}")
        else:
            logger.info(f"✗ 路径不存在")

if __name__ == "__main__":
    logger.info("=== Reddit评论情感分析程序 (使用DeepSeek API) ===")
    logger.info("开始时间: " + time.strftime("%Y-%m-%d %H:%M:%S"))
    
    # 可选：测试环境和文件路径
    # test_environment()
    
    # 运行主程序
    main()
    
    logger.info("\n程序执行完成！")
    logger.info("结束时间: " + time.strftime("%Y-%m-%d %H:%M:%S"))

2026-01-20 01:10:39,275 - INFO - === Reddit评论情感分析程序 (使用DeepSeek API) ===
2026-01-20 01:10:39,277 - INFO - 开始时间: 2026-01-20 01:10:39
2026-01-20 01:10:39,278 - INFO - 正在查找Excel文件...
2026-01-20 01:10:39,279 - INFO - 找到文件: /Users/griffinx/Desktop/Maison/2. Antigravity/上市后监测/python 批量判定情感度/有效数据.xlsx
2026-01-20 01:10:39,279 - INFO - 使用文件: /Users/griffinx/Desktop/Maison/2. Antigravity/上市后监测/python 批量判定情感度/有效数据.xlsx
2026-01-20 01:10:39,280 - INFO - 正在读取Excel文件...
2026-01-20 01:10:39,361 - INFO - 数据形状: (895, 2)
2026-01-20 01:10:39,362 - INFO - 列名: ['帖子/评论内容', '帖子/评论内容_中文翻译']
2026-01-20 01:10:39,362 - INFO - 
前5行数据预览:
2026-01-20 01:10:39,363 - INFO - 行1: A列='Has anyone tried flying the Antigravity A1 indoors...' B列='有人尝试过在室内驾驶Antigravity A1吗？我在 Youtube 上观看了大量有关新 A1 ...'
2026-01-20 01:10:39,364 - INFO - 行2: A列='Rumors are heavily suggesting the imminent release...' B列='有传言称大疆即将发布 Avata 360，也许值得等待，看看该型号与 A1 的对比如何。 360 有...'
2026-01-20 01:10:39,364 - INFO - 行3: A列='I've heard those rumors as well a